# **Pipeline de Coleta e Visualização de POIs (Estações de Metro da Maia)**


In [ ]:
# -*- coding: utf-8 -*-
"""Pipeline de Reconciliação - Metro (OSM vs GTFS vs WFS vs OverTure)

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Rc1AaOzQSwrWFVJWQzomUrCGvJRFgbv0

# **Pipeline de Coleta e Visualização de POIs (Estações de Metro da Maia)**
"""

# --- 1. INSTALAÇÃO E IMPORTAÇÕES ---
print("Instalando bibliotecas...")
!pip install geopandas folium shapely gspread google-auth google-auth-oauthlib google-auth-httplib2 ipynbname duckdb boto3 botocore unidecode geopy fuzzywuzzy python-Levenshtein awscli --upgrade google-colab -q
print("Bibliotecas instaladas.")

# Configuração AWS
!aws configure set s3.signature_version s3v4
!aws configure set region us-west-2

import os
import json
import time
import datetime
import math
import smtplib
import zipfile
import warnings
from io import BytesIO
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from unidecode import unidecode

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point, mapping
import numpy as np
import folium
import ipynbname
import duckdb
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from shapely import wkb
from branca.element import MacroElement
from jinja2 import Template

from google.colab import drive
from google.colab import userdata
import gspread
from google.colab import auth as colab_auth
from google.auth import default as google_auth_default

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- 2. CONFIGURAÇÕES ---
profiling_log = []
script_start_time = time.time()

def start_timer(): return time.time()
def log_time(task, start_t):
    elapsed_proc = time.time() - start_t
    elapsed_watch = time.time() - script_start_time
    print(f"[{task}] {elapsed_proc:.2f}s")
    profiling_log.append({"task": task, "proc_time": elapsed_proc, "watch_time": elapsed_watch})

try:
    drive.mount('/content/drive')
    base_path_secret = userdata.get('DRIVE_SAVE_PATH')
    DRIVE_SAVE_PATH = base_path_secret if base_path_secret else "/content/drive/MyDrive/Colab_Pipelines/Maia_Metro/"
    if not os.path.exists(DRIVE_SAVE_PATH): os.makedirs(DRIVE_SAVE_PATH)
except Exception as e:
    print(f"Aviso Drive: {e}")
    DRIVE_SAVE_PATH = "/content/"

try:
    GMAIL_USER = userdata.get('GMAIL_USER')
    GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
    email_list_raw = userdata.get('DESTINATION_EMAIL_LIST')
    if "," in email_list_raw:
        DESTINATION_EMAIL_LIST = [e.strip() for e in email_list_raw.split(',')]
    else:
        DESTINATION_EMAIL_LIST = [email_list_raw.strip()]
except:
    GMAIL_USER = None
    DESTINATION_EMAIL_LIST = []

MAIA_BOUNDARY_URL = "https://baze.cm-maia.pt/BaZe/api/api4gj.php?nome=limconcvm"
GTFS_FILE_PATH = "/content/drive/MyDrive/google_transit.zip"
MAIA_WFS_URL = "https://geoserver.cm-maia.pt/geoserver/amp/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=amp:Estações%20do%20Metro&maxFeatures=5000&outputFormat=application%2Fjson"
OVERTURE_S3_PATH = "s3://overturemaps-us-west-2/release/2024-11-13.0/theme=places/type=place/*.parquet"

OVERPASS_URLS = [
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]
OSM_TAG_QUERY = '["railway"="station"]["station"~"light_rail|subway"]'
API_TIMEOUT = 90
QUERY_BUFFER = 0.01
DISTANCE_THRESHOLD_METERS = 30

GOOGLE_SHEET_NAME = f"Reconciliacao_Metro_Maia_Full_{datetime.date.today().isoformat()}"
HTML_MAP_FILENAME = os.path.join(DRIVE_SAVE_PATH, f"mapa_reconciliacao_metro_{datetime.date.today().isoformat()}.html")

# --- ESTILOS VISUAIS ---
STYLE_CONFIG = {
    "Total": {"folium": "green", "hex": "green", "icon": "star", "label": "Match Perfeito (3 Fontes)"},
    "Parcial": {"folium": "lightgreen", "hex": "#90EE90", "icon": "check", "label": "Match Parcial"},
    "ConflitoLoc": { "folium": "orange", "hex": "#FFA500", "icon": "arrows-alt", "label": "Conflito Localização (>30m)" },
    "ConflitoNome": { "folium": "blue", "hex": "#1E90FF", "icon": "font", "label": "Conflito de Nomes" },
    "Apenas": {"folium": "purple", "hex": "purple", "icon": "question", "label": "Apenas no OSM"},
    "Default": {"folium": "gray", "hex": "gray", "icon": "info", "label": "Indefinido"}
}

context_vars = {'script_name': ipynbname.name() + ".ipynb", 'date': str(datetime.date.today())} if 'ipynbname' in locals() else {}

# --- 3. CARREGAMENTO ---

def normalize_text(text):
    if not text: return ""
    return unidecode(str(text)).strip().lower()

def approx_distance_meters(lat1, lon1, lat2, lon2):
    try:
        m_per_deg_lat = 111132.954
        m_per_deg_lon = 111319.488 * np.cos(np.radians((lat1 + lat2) / 2))
        dx = (lon2 - lon1) * m_per_deg_lon
        dy = (lat2 - lat1) * m_per_deg_lat
        return np.sqrt(dx**2 + dy**2)
    except: return float('inf')

def load_maia_boundary(url):
    s = start_timer()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = requests.get(url, verify=False, timeout=30)
            maia_gdf = gpd.read_file(BytesIO(r.content))

            if maia_gdf.empty: return None
            maia_boundary = maia_gdf.geometry.unary_union

            if maia_boundary.geom_type == "Polygon": return maia_boundary
            elif maia_boundary.geom_type == "MultiPolygon": return max(maia_boundary.geoms, key=lambda p: p.area)
            elif maia_boundary.geom_type == "LineString":
                coords = list(maia_boundary.coords)
                if coords[0] != coords[-1]: coords.append(coords[0])
                return Polygon(coords)

            log_time("Limite Maia", s)
            return maia_boundary
    except Exception as e:
        print(f"Erro limite: {e}")
        return None

def query_osm_data(polygon, osm_tag_query, timeout_seconds, buffer, urls_list):
    s = start_timer()
    if polygon is None: return []
    minx, miny, maxx, maxy = polygon.bounds
    query = f"""
    [out:json][timeout:{timeout_seconds}];
    (
      node{osm_tag_query}({miny-buffer},{minx-buffer},{maxy+buffer},{maxx+buffer});
      way{osm_tag_query}({miny-buffer},{minx-buffer},{maxy+buffer},{maxx+buffer});
      relation{osm_tag_query}({miny-buffer},{minx-buffer},{maxy+buffer},{maxx+buffer});
    );
    out center tags;
    """
    for url in urls_list:
        try:
            # ✅ CORREÇÃO: Headers iguais ao openstreetmap-feature-extractor
            headers = {'Content-Type': 'text/plain'}
            r = requests.post(url, data=query.strip().encode('utf-8'), headers=headers, timeout=timeout_seconds)
            if r.status_code == 200:
                data = r.json().get("elements", [])
                log_time(f"Overpass OSM ({len(data)})", s)
                return data
        except: continue
    return []

def process_osm_elements(elements):
    features = []
    if not elements: return gpd.GeoDataFrame()
    for e in elements:
        lat, lon = None, None
        if e['type'] == 'node': lat, lon = e.get("lat"), e.get("lon")
        elif "center" in e: lat, lon = e["center"]["lat"], e["center"]["lon"]
        if lat and lon:
            tags = e.get("tags", {})
            features.append({
                "Name": tags.get("name", "Sem Nome"),
                "Type": "Metro",
                "Latitude": lat, "Longitude": lon,
                "OSM_ID": e.get("id"),
                "OSM_Type": e.get("type"),
                "geometry": Point(lon, lat), "Tags": json.dumps(tags)
            })
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def filter_points_in_polygon(gdf, poly):
    if gdf.empty or not poly: return gpd.GeoDataFrame()
    if gdf.crs is None: gdf.set_crs(epsg=4326, inplace=True)
    elif gdf.crs != "EPSG:4326": gdf = gdf.to_crs("EPSG:4326")
    return gdf[gdf.within(poly)].copy()

def load_gtfs_stops_from_zip(zip_path):
    s = start_timer()
    try:
        if not os.path.exists(zip_path): return gpd.GeoDataFrame()
        z = zipfile.ZipFile(zip_path, 'r')
        with z.open('stops.txt') as f: df = pd.read_csv(f)
        geom = [Point(xy) for xy in zip(df['stop_lon'], df['stop_lat'])]
        df['stop_name_norm'] = df['stop_name'].apply(normalize_text)
        log_time(f"GTFS Local ({len(df)})", s)
        return gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326")
    except Exception as e:
        print(f"Erro GTFS: {e}")
        return gpd.GeoDataFrame()

def load_wfs_maia_data(url):
    s = start_timer()
    try:
        r = requests.get(url, verify=False, timeout=60)
        gdf = gpd.read_file(BytesIO(r.content))
        if gdf.empty: return gpd.GeoDataFrame()
        if gdf.crs is None: gdf.set_crs(epsg=3763, inplace=True, allow_override=True)
        gdf = gdf.to_crs(epsg=4326)
        if '04_ESTACAO' in gdf.columns: gdf['Name'] = gdf['04_ESTACAO']
        else: gdf['Name'] = "Desconhecido"
        gdf['Name_Norm'] = gdf['Name'].apply(normalize_text)
        log_time(f"WFS Maia ({len(gdf)})", s)
        return gdf
    except Exception as e:
        print(f"Erro WFS: {e}")
        return gpd.GeoDataFrame()

def load_overture_places(polygon, s3_path_pattern):
    s = start_timer()
    try:
        minx, miny, maxx, maxy = polygon.bounds
        bucket_name = "overturemaps-us-west-2"
        s3 = boto3.client('s3', region_name='us-west-2', config=Config(signature_version=UNSIGNED))

        try:
            response = s3.list_objects_v2(Bucket=bucket_name, Prefix='release/', Delimiter='/')
            versions = [p['Prefix'].split('/')[-2] for p in response.get('CommonPrefixes', [])]
            versions.sort()
            latest_version = versions[-1] if versions else "2024-11-13.0"
            prefix = f"release/{latest_version}/theme=places/type=place/"
        except: prefix = "release/2024-11-13.0/theme=places/type=place/"

        file_urls = []
        paginator = s3.get_paginator('list_objects_v2')
        count = 0; limit = 500
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            if 'Contents' in page:
                for obj in page['Contents']:
                    if obj['Key'].endswith('.parquet'):
                        file_urls.append(f"https://{bucket_name}.s3.us-west-2.amazonaws.com/{obj['Key']}")
                        count += 1
            if count >= limit: break

        if not file_urls: return gpd.GeoDataFrame()

        con = duckdb.connect()
        con.execute("INSTALL spatial; LOAD spatial;")
        con.execute("INSTALL httpfs; LOAD httpfs;")

        query = f"""
        SELECT names.primary as Name, categories.primary as Category, ST_AsWKB(geometry) as geometry
        FROM read_parquet({file_urls}, filename=true, hive_partitioning=1)
        WHERE bbox.xmin > {minx} AND bbox.xmax < {maxx} AND bbox.ymin > {miny} AND bbox.ymax < {maxy}
        AND geometry IS NOT NULL
        AND (categories.primary IN ('subway_station', 'light_rail_station', 'train_station', 'tram_stop', 'bus_station', 'public_transport_stop')
             OR names.primary ILIKE '%Metro%' OR names.primary ILIKE '%Estação%')
        """
        df = con.execute(query).df()
        if df.empty: return gpd.GeoDataFrame()

        def robust_wkb_load(x):
            try: return wkb.loads(bytes(x)) if isinstance(x, (bytes, bytearray)) else None
            except: return None

        df['geometry'] = df['geometry'].apply(robust_wkb_load)
        df = df.dropna(subset=['geometry'])

        blacklisted = ['restaurant', 'barbecue_restaurant', 'bakery', 'gas_station', 'school', 'grocery_store', 'hair_salon', 'beauty_salon']
        df_clean = df[~df['Category'].isin(blacklisted)].copy()

        def clean_station_name(name):
            if not name: return "Sem Nome"
            n = str(name).strip()
            for p in ["Apeadeiro de ", "Apeadeiro do ", "Estação Ferroviária de ", "Estação de ", "Paragem de ", "Metro "]:
                if n.lower().startswith(p.lower()): n = n[len(p):]
            return n

        df_clean['Name'] = df_clean['Name'].apply(clean_station_name)
        df_clean['Name_Norm'] = df_clean['Name'].apply(normalize_text)

        gdf = gpd.GeoDataFrame(df_clean, geometry='geometry', crs="EPSG:4326")
        gdf = gdf[gdf.within(polygon)]
        log_time(f"Overture ({len(gdf)})", s)
        return gdf

    except Exception as e:
        print(f"Erro Overture: {e}")
        return gpd.GeoDataFrame()

# --- 4. RECONCILIAÇÃO ---
def reconcile_data(gdf_osm, gdf_gtfs, gdf_wfs, gdf_overture, threshold_meters):
    s = start_timer()
    print(f"A reconciliar {len(gdf_osm)} OSM vs External Sources...")
    comparison_data = []

    if gdf_osm.empty: return gpd.GeoDataFrame()

    for i, osm_row in gdf_osm.iterrows():
        osm_name_raw = str(osm_row.get('Name', '')).strip()
        osm_name_norm = normalize_text(osm_name_raw)
        osm_lat, osm_lon = osm_row.geometry.y, osm_row.geometry.x

        def get_best_match(source_gdf, name_col_norm, name_col_orig):
            obj = {"name": "-", "dist": float('inf'), "lat": "-", "lon": "-", "match_name": False}
            if source_gdf.empty: return obj

            matches = source_gdf[source_gdf[name_col_norm] == osm_name_norm]
            if matches.empty:
                matches = source_gdf[source_gdf[name_col_norm].str.contains(osm_name_norm, regex=False) |
                                     source_gdf[name_col_norm].apply(lambda x: osm_name_norm in x)]

            if not matches.empty:
                dists = matches.apply(lambda r: approx_distance_meters(osm_lat, osm_lon, r.geometry.y, r.geometry.x), axis=1)
                best_idx = dists.idxmin()
                row = matches.loc[best_idx]
                obj.update({"dist": dists[best_idx], "name": row[name_col_orig], "lat": row.geometry.y, "lon": row.geometry.x, "match_name": True})
            else:
                dists = source_gdf.apply(lambda r: approx_distance_meters(osm_lat, osm_lon, r.geometry.y, r.geometry.x), axis=1)
                best_idx = dists.idxmin()
                row = source_gdf.loc[best_idx]
                found_norm = normalize_text(row[name_col_orig])
                match_name = (osm_name_norm in found_norm or found_norm in osm_name_norm)
                obj.update({"dist": dists[best_idx], "name": row[name_col_orig], "lat": row.geometry.y, "lon": row.geometry.x, "match_name": match_name})
            return obj

        gtfs_obj = get_best_match(gdf_gtfs, 'stop_name_norm', 'stop_name')
        wfs_obj = get_best_match(gdf_wfs, 'Name_Norm', 'Name')
        ov_obj = get_best_match(gdf_overture, 'Name_Norm', 'Name')

        if ov_obj["dist"] != float('inf') and ov_obj["dist"] > 50 and not ov_obj["match_name"]:
            ov_obj = {"name": "-", "dist": float('inf'), "lat": "-", "lon": "-", "match_name": False}

        good_matches_count = 0
        loc_conflicts = []
        name_conflicts = []

        for src, obj in [("GTFS", gtfs_obj), ("WFS", wfs_obj), ("Overture", ov_obj)]:
            if obj["dist"] != float('inf'):
                if obj["match_name"] and obj["dist"] <= threshold_meters:
                    good_matches_count += 1
                elif obj["match_name"] and obj["dist"] > threshold_meters:
                    loc_conflicts.append(f"{src} ({int(obj['dist'])}m)")
                elif not obj["match_name"] and obj["dist"] <= threshold_meters:
                    name_conflicts.append(f"{src} ({obj['name']})")

        if len(loc_conflicts) > 0:
            status_text = f"Conflito Local: {', '.join(loc_conflicts)}"
            status_code = "ConflitoLoc"
        elif len(name_conflicts) > 0:
            status_text = f"Conflito Nome: {', '.join(name_conflicts)}"
            status_code = "ConflitoNome"
        elif good_matches_count == 3:
            status_text = "Match Total (3 Fontes)"
            status_code = "Total"
        elif good_matches_count > 0:
            status_text = "Match Parcial (Faltam Fontes)"
            status_code = "Parcial"
        else:
            status_text = "Apenas no OSM"
            status_code = "Apenas"

        def fmt_dist(d): return d if d != float('inf') and d != '-' else None

        comparison_data.append({
            'Nome_OSM': osm_name_raw, 'OSM_ID': osm_row.get('OSM_ID', ''), 'OSM_Type': osm_row.get('OSM_Type', 'node'),
            'Status': status_text,
            'Code': status_code,
            'Lat_OSM': osm_lat, 'Lon_OSM': osm_lon,
            'Nome_GTFS': gtfs_obj["name"], 'Distancia_GTFS': fmt_dist(gtfs_obj["dist"]), 'Lat_GTFS': gtfs_obj["lat"], 'Lon_GTFS': gtfs_obj["lon"],
            'Nome_WFS': wfs_obj["name"], 'Distancia_WFS': fmt_dist(wfs_obj["dist"]), 'Lat_WFS': wfs_obj["lat"], 'Lon_WFS': wfs_obj["lon"],
            'Nome_Overture': ov_obj["name"], 'Distancia_Overture': fmt_dist(ov_obj["dist"]), 'Lat_Overture': ov_obj["lat"], 'Lon_Overture': ov_obj["lon"],
            'geometry': osm_row.geometry
        })

    log_time("Reconciliação Lógica", s)
    return gpd.GeoDataFrame(comparison_data, crs="EPSG:4326")

def save_gdf_to_google_sheet(gdf, sheet_name):
    print("A guardar no Google Sheets...")
    if gdf.empty: return "#"
    try:
        colab_auth.authenticate_user()
        creds, _ = google_auth_default()
        gc = gspread.authorize(creds)
        try: sh = gc.open(sheet_name)
        except: sh = gc.create(sheet_name)

        for email in DESTINATION_EMAIL_LIST:
            try: sh.share(email, perm_type='user', role='writer')
            except: pass

        ws = sh.get_worksheet(0)
        ws.clear()
        df = gdf.drop(columns=['geometry'], errors='ignore').copy()
        df = df.fillna('').astype(str)
        ws.update([df.columns.values.tolist()] + df.values.tolist(), 'A1')
        return f"https://docs.google.com/spreadsheets/d/{sh.id}"
    except Exception as e:
        print(f"Erro Sheets: {e}")
        return "#"

# --- 5. MAPA ---
class MapLegend(MacroElement):
    _template = Template(u"""
    {% macro html(this, kwargs) %}
    <div style="position: fixed; bottom: 30px; right: 30px; width: 340px; z-index:9999; border: 2px solid rgba(0,0,0,0.2); border-radius: 8px; background-color: rgba(255, 255, 255, 0.95); padding: 15px; font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <h4 style="margin-top:0; margin-bottom: 12px; font-size: 16px; color: #333; border-bottom: 1px solid #ddd; padding-bottom: 8px;"><b>{{ this.title }}</b></h4>
    <table style="width:100%; border-collapse: collapse;">
        {% for label, config in this.items %}
        <tr style="margin-bottom: 8px; border-bottom: 1px solid #f0f0f0;">
            <td style="width: 30px; text-align: center; padding: 6px;"><i class="fa fa-{{ config.icon }}" style="color: {{ config.hex }}; font-size: 16px;"></i></td>
            <td style="padding: 6px; color: #444;">{{ label }}</td>
        </tr>
        {% endfor %}
    </table>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px;">
        <small style="color:#555; font-weight:bold;">Linhas de Desvio (Conflitos):</small>
        <div style="font-size: 11px; margin-top:4px;">
            <span style="color:gray; font-weight:bold; font-size: 14px;">---</span> GTFS
            <span style="margin-left:8px; color:black; font-weight:bold; font-size: 14px;">---</span> WFS
            <span style="margin-left:8px; color:cyan; font-weight:bold; font-size: 14px;">---</span> Overture
        </div>
    </div>

    <div style="margin-top: 10px; border-top: 1px solid #eee; padding-top: 8px; text-align: center;">
        <small style="color:#555; font-weight:bold; display:block; margin-bottom:4px;">Fontes Oficiais</small>
        <div style="font-size: 11px; color: #444;">
            <a href="https://www.metrodoporto.pt" target="_blank" style="color:#0078A8; text-decoration:none;">MdP (GTFS)</a>
            <span style="color:#ccc; margin:0 3px;">|</span>
            <a href="https://geoserver.cm-maia.pt" target="_blank" style="color:#0078A8; text-decoration:none;">Maia (WFS)</a>
            <span style="color:#ccc; margin:0 3px;">|</span>
            <a href="https://overturemaps.org" target="_blank" style="color:#0078A8; text-decoration:none;">Overture</a>
            <span style="color:#ccc; margin:0 3px;">|</span>
            <a href="https://www.openstreetmap.org" target="_blank" style="color:#0078A8; text-decoration:none;">OSM</a>
        </div>
    </div>
    </div>
    {% endmacro %}
    """)
    def __init__(self, title, items):
        super(MapLegend, self).__init__()
        self._name = 'MapLegend'
        self.title = title
        self.items = items

def create_map(gdf, poly_geom, filename):
    print("Gerando Mapa...")
    center = [poly_geom.centroid.y, poly_geom.centroid.x] if poly_geom else [41.23, -8.62]
    m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")

    if poly_geom is not None:
        try:
            folium.GeoJson(data=mapping(poly_geom), name="Limite Maia", style_function=lambda x: {'color': '#000000', 'weight': 3, 'fillOpacity': 0, 'dashArray': '5, 5'}).add_to(m)
        except: pass

    def is_valid(lat, lon):
        try:
            if lat is None or lon is None: return False
            if str(lat) == '-' or str(lon) == '-': return False
            if math.isnan(float(lat)) or math.isnan(float(lon)): return False
            return True
        except: return False

    def fmt_d(v): return f"{float(v):.1f}m" if is_valid(v, 0) else "-"

    if not gdf.empty:
        for idx, r in gdf.iterrows():
            try:
                if not is_valid(r['Lat_OSM'], r['Lon_OSM']): continue

                code = r.get('Code', 'Default')
                config = STYLE_CONFIG.get(code, STYLE_CONFIG['Default'])

                popup_html = f"""
                <div style='font-family: "Segoe UI", sans-serif; width: 380px; font-size: 13px;'>
                    <div style='background-color: {config['hex']}; color: {'white' if 'Conflito' in code else 'black'}; padding: 6px 10px; border-radius: 4px;'><b>{r['Status']}</b></div>
                    <div style="margin-top: 8px;"><b style='color:#333; font-size:15px;'>🌍 {r['Nome_OSM']} (OSM)</b></div>
                    <div style='margin-top:8px; padding: 8px; background-color: #f0f7fb; border-left: 3px solid #0078A8;'><b>📍 Coords OSM:</b> {r['Lat_OSM']:.5f}, {r['Lon_OSM']:.5f}</div>
                    <hr>
                    <table style='width:100%; font-size:12px;'>
                        <tr><td><b>🚇 GTFS:</b></td><td>{r['Nome_GTFS']}</td><td>{fmt_d(r['Distancia_GTFS'])}</td></tr>
                        <tr><td><b>🏛️ WFS:</b></td><td>{r['Nome_WFS']}</td><td>{fmt_d(r['Distancia_WFS'])}</td></tr>
                        <tr><td><b>🏢 OV:</b></td><td>{r['Nome_Overture']}</td><td>{fmt_d(r['Distancia_Overture'])}</td></tr>
                    </table>
                </div>"""

                lat_osm, lon_osm = float(r['Lat_OSM']), float(r['Lon_OSM'])

                folium.Marker([lat_osm, lon_osm], popup=folium.Popup(popup_html, max_width=400), icon=folium.Icon(color=config['folium'], icon=config['icon'], prefix='fa')).add_to(m)

                if is_valid(r['Lat_GTFS'], r['Lon_GTFS']):
                    folium.PolyLine([[lat_osm, lon_osm], [float(r['Lat_GTFS']), float(r['Lon_GTFS'])]], color="gray", weight=2, dash_array="5,5").add_to(m)

                if is_valid(r['Lat_WFS'], r['Lon_WFS']):
                    folium.PolyLine([[lat_osm, lon_osm], [float(r['Lat_WFS']), float(r['Lon_WFS'])]], color="black", weight=2, dash_array="5,5").add_to(m)

                if is_valid(r['Lat_Overture'], r['Lon_Overture']):
                    folium.PolyLine([[lat_osm, lon_osm], [float(r['Lat_Overture']), float(r['Lon_Overture'])]], color="cyan", weight=2, dash_array="5,5").add_to(m)
            except: pass

    items = [(v['label'], v) for k, v in STYLE_CONFIG.items() if k != "Default"]
    m.add_child(MapLegend("Reconciliação Metro", items))
    m.save(filename)
    return m

# --- 6. EMAIL ---
def send_execution_email(sheet_url):
    if not GMAIL_USER or not DESTINATION_EMAIL_LIST: return

    print("📧 A enviar email de relatório...")
    log_rows = ""
    for entry in profiling_log:
        log_rows += f"""<tr><td style="border:1px solid black;padding:4px;">{entry['task']}</td><td style="border:1px solid black;padding:4px;text-align:right;">{entry['proc_time']:.2f}</td></tr>"""

    html_body = f"""
    <html><body style="font-family: Arial;">
        <h2>Pipeline Metro Maia</h2>
        <p>Google Sheet: <a href="{sheet_url}">{sheet_url}</a></p>
        <table style="border-collapse:collapse;width:100%;border:2px solid black;">
            <thead><tr style="background:#f2f2f2;"><th>Task</th><th>Seconds</th></tr></thead>
            <tbody>{log_rows}</tbody>
        </table>
    </body></html>
    """

    msg = MIMEMultipart()
    msg['From'], msg['To'], msg['Subject'] = GMAIL_USER, ", ".join(DESTINATION_EMAIL_LIST), f"Pipeline Log: Maia Metro {datetime.date.today()}"
    msg.attach(MIMEText(html_body, 'html'))

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587); server.starttls()
        server.login(GMAIL_USER, GMAIL_APP_PASSWORD)
        server.sendmail(GMAIL_USER, DESTINATION_EMAIL_LIST, msg.as_string())
        server.quit()
        print("✅ Email enviado.")
    except Exception as e: print(f"❌ Erro email: {e}")

# --- 7. EXECUÇÃO ---
print("\n--- INICIO PIPELINE METRO ---")

maia_polygon = load_maia_boundary(MAIA_BOUNDARY_URL)

if maia_polygon:
    gdf_gtfs = load_gtfs_stops_from_zip(GTFS_FILE_PATH)
    gdf_wfs = load_wfs_maia_data(MAIA_WFS_URL)
    gdf_overture = load_overture_places(maia_polygon, OVERTURE_S3_PATH)

    osm_data = query_osm_data(maia_polygon, OSM_TAG_QUERY, API_TIMEOUT, QUERY_BUFFER, OVERPASS_URLS)
    gdf_osm_all = process_osm_elements(osm_data)

    print(f"DEBUG: OSM Bruto: {len(gdf_osm_all)}")
    gdf_osm_maia = filter_points_in_polygon(gdf_osm_all, maia_polygon)
    print(f"DEBUG: OSM Maia: {len(gdf_osm_maia)}")

    if not gdf_osm_maia.empty:
        gdf_reconciled = reconcile_data(gdf_osm_maia, gdf_gtfs, gdf_wfs, gdf_overture, DISTANCE_THRESHOLD_METERS)
        sheet_url = save_gdf_to_google_sheet(gdf_reconciled, GOOGLE_SHEET_NAME)
        folium_map = create_map(gdf_reconciled, maia_polygon, HTML_MAP_FILENAME)
        print(f"✅ Mapa guardado: {HTML_MAP_FILENAME}")
        send_execution_email(sheet_url)
    else: print("❌ OSM Vazio na Maia.")
else: print("❌ Falha Limite Maia.")

Instalando bibliotecas...
Bibliotecas instaladas.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- INICIO PIPELINE METRO ---
[GTFS Local (85)] 0.01s
[WFS Maia (24)] 1.14s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[Overture (34)] 13.20s
[Overpass OSM (48)] 6.97s
DEBUG: OSM Bruto: 48
DEBUG: OSM Maia: 12
A reconciliar 12 OSM vs External Sources...
[Reconciliação Lógica] 0.07s
A guardar no Google Sheets...
Gerando Mapa...
✅ Mapa guardado: /content/drive/MyDrive/Colab_Pipelines/Maia_Metro/mapa_reconciliacao_metro_2026-01-21.html
📧 A enviar email de relatório...
✅ Email enviado.


In [ ]:
display(folium_map)